# Data API Collector — Databricks Integration Example

This notebook demonstrates how to connect to the Data API Collector stack from Databricks:
1. **Kafka Streaming** — Read real-time synthetic data (fraud, telemetry, web traffic)
2. **SLED Data Generation** — Populate Neo4j, Postgres, and Kafka with State/Local/Higher Ed data
3. **SLED Kafka Streaming** — Read SLED events from Kafka (enrollment, grants, 311, K-12, procurement, case management)
4. **SLED Neo4j Queries** — Query SLED graph data (relationships, pathways, networks)
5. **SLED PostgreSQL Reads** — Read SLED relational tables via JDBC
6. **Neo4j** — General graph queries via Bolt
7. **PostgreSQL** — Read tables via JDBC
8. **REST API** — Health checks and service interaction

## Prerequisites
- Data API Collector stack running with generators active
- Network access from Databricks to your host (nginx TCP forwarding on ports 9093, 7688)
- Databricks secrets configured (see Setup cell below)

In [ ]:
%pip install neo4j
dbutils.library.restartPython()

## Setup — Secrets & Configuration

Store your credentials using Databricks secrets before running this notebook:

```bash
databricks secrets create-scope data-api
databricks secrets put-secret data-api api-key --string-value "YOUR_SECRET_KEY"
databricks secrets put-secret data-api kafka-user --string-value "YOUR_KAFKA_USER"
databricks secrets put-secret data-api kafka-password --string-value "YOUR_KAFKA_PASSWORD"
databricks secrets put-secret data-api neo4j-password --string-value "YOUR_NEO4J_PASSWORD"
databricks secrets put-secret data-api postgres-password --string-value "YOUR_POSTGRES_PASSWORD"
```

In [ ]:
import neo4j
import requests
from pyspark.sql.types import *
from pyspark.sql.functions import *

# ---------------------------------------------------------------------------
# Configuration — UPDATE THESE for your environment
# ---------------------------------------------------------------------------
HOST = "your-hostname.com"                  # your nginx host

# Secrets — loaded from Databricks secret scope
API_KEY         = dbutils.secrets.get(scope="data-api", key="api-key")
KAFKA_USER      = dbutils.secrets.get(scope="data-api", key="kafka-user")
KAFKA_PASSWORD  = dbutils.secrets.get(scope="data-api", key="kafka-password")
NEO4J_USER      = "neo4j"
NEO4J_PASSWORD  = dbutils.secrets.get(scope="data-api", key="neo4j-password")
POSTGRES_USER   = "postgres"
POSTGRES_PASSWORD = dbutils.secrets.get(scope="data-api", key="postgres-password")

# Derived URLs
API_URL       = f"https://{HOST}/api/v1"
KAFKA_BROKER  = f"{HOST}:9093"
NEO4J_URI     = f"bolt+s://{HOST}:7688"
POSTGRES_JDBC = f"jdbc:postgresql://{HOST}:15433/datacollector?ssl=true&sslmode=require"
HEADERS       = {"X-Api-Key": API_KEY}

# Checkpoint base path — update to your catalog/schema
CHECKPOINT_BASE = "/Volumes/your_catalog/your_schema/checkpoints"

# SLED use cases
SLED_USE_CASES = [
    "student_enrollment", "grant_budget", "citizen_services",
    "k12_early_warning", "procurement", "case_management",
]

---
## 1. REST API — Start Kafka Generators

Start the streaming data generators on the server. These produce synthetic data to Kafka topics.

In [ ]:
# Start all three core generators
for use_case in ["fraud_detection", "telemetry", "web_traffic"]:
    resp = requests.post(f"{API_URL}/kafka/generators/start", headers=HEADERS, json={
        "use_case": use_case,
        "rows_per_batch": 100,
        "batch_interval_seconds": 1.0,
        "timeout_minutes": 10,
    })
    data = resp.json()
    print(f"{use_case}: id={data.get('generator_id')} status={data.get('status')}")

# Check running generators
generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
print(f"\n{len(generators)} generator(s) running")

---
## 2. Kafka Streaming — Fraud Detection

Read the `streaming-fraud-transactions` topic as a Structured Streaming DataFrame.

In [ ]:
fraud_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("merchant_id", IntegerType()),
    StructField("amount", DecimalType(10, 2)),
    StructField("currency", StringType()),
    StructField("merchant_category", StringType()),
    StructField("payment_method", StringType()),
    StructField("ip_address", StringType()),
    StructField("device_id", StringType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("card_type", StringType()),
    StructField("event_timestamp", StringType()),
])

raw_fraud = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-fraud-transactions")
    .option("startingOffsets", "earliest")
    .load()
)

fraud_parsed = (
    raw_fraud
    .select(from_json(col("value").cast("string"), fraud_schema).alias("data"))
    .select("data.*")
)

In [ ]:
# Preview the fraud stream — clear checkpoint first if restarting
dbutils.fs.rm(f"{CHECKPOINT_BASE}/fraud_preview", recurse=True)
display(fraud_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/fraud_preview")

### Write fraud stream to Delta table (optional)

In [ ]:
# Uncomment to write the fraud stream to a Delta table
# (
#     fraud_parsed.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("checkpointLocation", f"{CHECKPOINT_BASE}/fraud_transactions")
#     .toTable("your_catalog.your_schema.fraud_transactions")
# )

---
## 3. Kafka Streaming — Telemetry

Read the `streaming-device-telemetry` topic.

In [ ]:
telemetry_schema = StructType([
    StructField("event_id", StringType()),
    StructField("device_id", StringType()),
    StructField("device_type", StringType()),
    StructField("reading_value", DecimalType(10, 4)),
    StructField("unit", StringType()),
    StructField("battery_level", DecimalType(5, 2)),
    StructField("signal_strength_dbm", IntegerType()),
    StructField("firmware_version", StringType()),
    StructField("facility_id", IntegerType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("anomaly_flag", BooleanType()),
    StructField("event_timestamp", StringType()),
])

raw_telemetry = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-device-telemetry")
    .option("startingOffsets", "earliest")
    .load()
)

telemetry_parsed = (
    raw_telemetry
    .select(from_json(col("value").cast("string"), telemetry_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/telemetry_preview", recurse=True)
display(telemetry_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/telemetry_preview")

---
## 4. Kafka Streaming — Web Traffic

Read the `streaming-web-traffic` topic.

In [ ]:
web_schema = StructType([
    StructField("event_id", StringType()),
    StructField("session_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("page_url", StringType()),
    StructField("referrer", StringType()),
    StructField("user_agent", StringType()),
    StructField("ip_address", StringType()),
    StructField("country", StringType()),
    StructField("device_type", StringType()),
    StructField("action", StringType()),
    StructField("duration_ms", IntegerType()),
    StructField("http_status", IntegerType()),
    StructField("event_timestamp", StringType()),
])

raw_web = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-web-traffic")
    .option("startingOffsets", "earliest")
    .load()
)

web_parsed = (
    raw_web
    .select(from_json(col("value").cast("string"), web_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/web_preview", recurse=True)
display(web_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/web_preview")

---
## 5. SLED — Populate Data (Neo4j, Postgres & Kafka)

SLED (State, Local & Higher Ed) use cases generate realistic mock data across three targets.

| Use Case | Kafka Topic | Neo4j Graph | Postgres Tables |
|---|---|---|---|
| `student_enrollment` | `streaming-student-enrollment` | Student→Course→Department→DegreeProgram | sled_students, sled_courses, sled_enrollment_events |
| `grant_budget` | `streaming-grant-budget` | FundingSource→Agency→Program→Vendor→LineItem | sled_budget_transactions, sled_agencies, sled_vendors |
| `citizen_services` | `streaming-citizen-services` | Citizen→ServiceRequest→Department, Asset→District | sled_service_requests, sled_citizens, sled_assets |
| `k12_early_warning` | `streaming-k12-early-warning` | K12Student→School→Teacher, Intervention→RiskIndicator | sled_k12_events, sled_k12_students, sled_schools |
| `procurement` | `streaming-procurement` | ProcAgency→Contract→ProcVendor, Lobbyist | sled_procurement_events, sled_contracts |
| `case_management` | `streaming-case-management` | Client→Case→Caseworker→HHSAgency→HHSProgram | sled_case_events, sled_clients, sled_cases |

In [ ]:
# --- Start all 6 SLED Kafka streaming generators ---
print("Starting SLED Kafka generators...")
for use_case in SLED_USE_CASES:
    resp = requests.post(f"{API_URL}/kafka/generators/start", headers=HEADERS, json={
        "use_case": use_case,
        "rows_per_batch": 100,
        "batch_interval_seconds": 1.0,
        "timeout_minutes": 10,
    })
    data = resp.json()
    print(f"  {use_case}: id={data.get('generator_id')} status={data.get('status')}")

# --- Populate Neo4j with SLED graph data ---
print("\nPopulating Neo4j with SLED graph data...")
for use_case in SLED_USE_CASES:
    resp = requests.post(
        f"{API_URL}/data-sources/neo4j/sled/{use_case}/populate",
        headers=HEADERS,
        json={"num_records": 5000},
    )
    data = resp.json()
    print(f"  {use_case}: job={data.get('job_id')} status={data.get('status')}")

# --- Populate Postgres with SLED relational data ---
print("\nPopulating Postgres with SLED relational data...")
for use_case in SLED_USE_CASES:
    resp = requests.post(
        f"{API_URL}/data-sources/sled/{use_case}/populate",
        headers=HEADERS,
        json={"num_records": 5000},
    )
    data = resp.json()
    print(f"  {use_case}: job={data.get('job_id')} status={data.get('status')}")

# Check all generators
generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
print(f"\n{len(generators)} total generator(s) running")

In [ ]:
# Check populate job status (Neo4j and Postgres run in background)
import time
import builtins
time.sleep(10)  # give jobs a moment to complete

print("Neo4j populate status:")
for use_case in SLED_USE_CASES:
    status = requests.get(f"{API_URL}/data-sources/neo4j/sled/{use_case}/status", headers=HEADERS).json()
    total = builtins.sum(v for k, v in status["counts"].items() if k != "_relationships" and isinstance(v, int))
    print(f"  {use_case}: {total} nodes")

print("\nPostgres populate status:")
for use_case in SLED_USE_CASES:
    status = requests.get(f"{API_URL}/data-sources/sled/{use_case}/status", headers=HEADERS).json()
    total = builtins.sum(v for v in status["counts"].values() if isinstance(v, int))
    print(f"  {use_case}: {total} rows")

---
## 6. SLED Kafka Streaming — Student Enrollment

Read the `streaming-student-enrollment` topic.

In [ ]:
student_enrollment_schema = StructType([
    StructField("event_id", StringType()),
    StructField("student_id", StringType()),
    StructField("course_id", StringType()),
    StructField("action", StringType()),
    StructField("semester", StringType()),
    StructField("department", StringType()),
    StructField("grade", StringType()),
    StructField("credits", IntegerType()),
    StructField("campus", StringType()),
    StructField("gpa_impact", DecimalType(3, 1)),
    StructField("event_timestamp", StringType()),
])

raw_enrollment = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-student-enrollment")
    .option("startingOffsets", "earliest")
    .load()
)

enrollment_parsed = (
    raw_enrollment
    .select(from_json(col("value").cast("string"), student_enrollment_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/enrollment_preview", recurse=True)
display(enrollment_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/enrollment_preview")

---
## 7. SLED Kafka Streaming — Grant & Budget

Read the `streaming-grant-budget` topic.

In [ ]:
grant_budget_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("fund_source_id", StringType()),
    StructField("agency_id", StringType()),
    StructField("program_id", StringType()),
    StructField("vendor_id", StringType()),
    StructField("transaction_type", StringType()),
    StructField("amount", DecimalType(10, 2)),
    StructField("fund_category", StringType()),
    StructField("fiscal_year", StringType()),
    StructField("quarter", StringType()),
    StructField("cost_center", StringType()),
    StructField("account_code", StringType()),
    StructField("description", StringType()),
    StructField("event_timestamp", StringType()),
])

raw_budget = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-grant-budget")
    .option("startingOffsets", "earliest")
    .load()
)

budget_parsed = (
    raw_budget
    .select(from_json(col("value").cast("string"), grant_budget_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/budget_preview", recurse=True)
display(budget_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/budget_preview")

---
## 8. SLED Kafka Streaming — Citizen Services (311)

Read the `streaming-citizen-services` topic.

In [ ]:
citizen_services_schema = StructType([
    StructField("request_id", StringType()),
    StructField("citizen_id", StringType()),
    StructField("request_type", StringType()),
    StructField("department", StringType()),
    StructField("status", StringType()),
    StructField("priority", StringType()),
    StructField("district", IntegerType()),
    StructField("asset_id", StringType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("response_time_hours", IntegerType()),
    StructField("satisfaction_rating", IntegerType()),
    StructField("event_timestamp", StringType()),
])

raw_311 = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-citizen-services")
    .option("startingOffsets", "earliest")
    .load()
)

citizen_parsed = (
    raw_311
    .select(from_json(col("value").cast("string"), citizen_services_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/citizen_services_preview", recurse=True)
display(citizen_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/citizen_services_preview")

---
## 9. SLED Kafka Streaming — K-12 Early Warning

Read the `streaming-k12-early-warning` topic.

In [ ]:
k12_schema = StructType([
    StructField("event_id", StringType()),
    StructField("student_id", StringType()),
    StructField("school_id", StringType()),
    StructField("event_type", StringType()),
    StructField("grade_level", IntegerType()),
    StructField("teacher_id", StringType()),
    StructField("risk_score", DecimalType(5, 2)),
    StructField("attendance_rate", DecimalType(5, 2)),
    StructField("gpa", DecimalType(3, 2)),
    StructField("behavior_incidents_ytd", IntegerType()),
    StructField("intervention_type", StringType()),
    StructField("school_type", StringType()),
    StructField("free_reduced_lunch", BooleanType()),
    StructField("english_learner", BooleanType()),
    StructField("special_education", BooleanType()),
    StructField("event_timestamp", StringType()),
])

raw_k12 = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-k12-early-warning")
    .option("startingOffsets", "earliest")
    .load()
)

k12_parsed = (
    raw_k12
    .select(from_json(col("value").cast("string"), k12_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/k12_preview", recurse=True)
display(k12_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/k12_preview")

---
## 10. SLED Kafka Streaming — Procurement

Read the `streaming-procurement` topic.

In [ ]:
procurement_schema = StructType([
    StructField("event_id", StringType()),
    StructField("agency_id", StringType()),
    StructField("vendor_id", StringType()),
    StructField("event_type", StringType()),
    StructField("contract_id", StringType()),
    StructField("amount", DecimalType(12, 2)),
    StructField("procurement_method", StringType()),
    StructField("commodity_code", StringType()),
    StructField("category", StringType()),
    StructField("minority_owned", BooleanType()),
    StructField("small_business", BooleanType()),
    StructField("local_vendor", BooleanType()),
    StructField("contract_duration_months", IntegerType()),
    StructField("payment_terms", StringType()),
    StructField("event_timestamp", StringType()),
])

raw_procurement = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-procurement")
    .option("startingOffsets", "earliest")
    .load()
)

procurement_parsed = (
    raw_procurement
    .select(from_json(col("value").cast("string"), procurement_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/procurement_preview", recurse=True)
display(procurement_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/procurement_preview")

---
## 11. SLED Kafka Streaming — Case Management (HHS)

Read the `streaming-case-management` topic.

In [ ]:
case_management_schema = StructType([
    StructField("event_id", StringType()),
    StructField("client_id", StringType()),
    StructField("case_id", StringType()),
    StructField("caseworker_id", StringType()),
    StructField("event_type", StringType()),
    StructField("program", StringType()),
    StructField("agency_id", StringType()),
    StructField("benefit_amount", DecimalType(10, 2)),
    StructField("household_size", IntegerType()),
    StructField("income_bracket", StringType()),
    StructField("county", StringType()),
    StructField("determination", StringType()),
    StructField("referral_source", StringType()),
    StructField("priority", StringType()),
    StructField("event_timestamp", StringType()),
])

raw_cases = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USER}" '
            f'password="{KAFKA_PASSWORD}";')
    .option("subscribe", "streaming-case-management")
    .option("startingOffsets", "earliest")
    .load()
)

cases_parsed = (
    raw_cases
    .select(from_json(col("value").cast("string"), case_management_schema).alias("data"))
    .select("data.*")
)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/case_management_preview", recurse=True)
display(cases_parsed, checkpointLocation=f"{CHECKPOINT_BASE}/case_management_preview")

---
## 12. SLED Neo4j — Graph Queries

Query the SLED graph data populated earlier. Each use case creates a rich network of nodes and relationships.

In [ ]:
driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, **kwargs):
    """Execute a Cypher query and return results as a list of dicts."""
    with driver.session() as session:
        result = session.run(query, **kwargs)
        return [record.data() for record in result]

# Overview: count all node labels
label_counts = run_cypher("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY count DESC
""")
df_labels = spark.createDataFrame(label_counts)
display(df_labels)

### Student Enrollment — Degree pathway analysis

In [ ]:
# Which departments have the most enrolled students?
dept_enrollment = run_cypher("""
    MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)-[:OFFERED_BY]->(d:Department)
    RETURN d.name AS department, count(DISTINCT s) AS students, count(c) AS enrollments
    ORDER BY students DESC
""")
display(spark.createDataFrame(dept_enrollment))

In [ ]:
# Course prerequisite chains — find courses with the deepest prereq trees
prereq_depth = run_cypher("""
    MATCH path = (c:Course)-[:REQUIRES*1..5]->(prereq:Course)
    RETURN c.name AS course, c.level AS level,
           length(path) AS prereq_depth,
           [n IN nodes(path) | n.name] AS chain
    ORDER BY prereq_depth DESC
    LIMIT 20
""")
display(spark.createDataFrame(prereq_depth))

### Grant & Budget — Fund flow analysis

In [ ]:
# Trace funding: source → agency → program → vendor
fund_flows = run_cypher("""
    MATCH (f:FundingSource)-[funds:FUNDS]->(a:Agency)-[:MANAGES]->(p:Program)-[:AWARDS]->(v:Vendor)
    RETURN f.name AS fund_source, f.category AS category,
           a.name AS agency, p.name AS program,
           v.name AS vendor, funds.amount AS amount
    ORDER BY funds.amount DESC
    LIMIT 25
""")
display(spark.createDataFrame(fund_flows))

In [ ]:
# Vendor subcontracting networks — find vendor clusters
subcontract_network = run_cypher("""
    MATCH (v1:Vendor)-[s:SUBCONTRACTS]->(v2:Vendor)
    RETURN v1.name AS prime_vendor, v2.name AS subcontractor,
           s.percentage AS subcontract_pct,
           v1.state AS prime_state, v2.state AS sub_state
    ORDER BY s.percentage DESC
    LIMIT 25
""")
display(spark.createDataFrame(subcontract_network))

### Citizen Services (311) — Service request analysis

In [ ]:
# Requests by district and priority
district_requests = run_cypher("""
    MATCH (sr:ServiceRequest)<-[:SUBMITTED]-(c:Citizen)
    RETURN sr.priority AS priority, count(sr) AS requests,
           collect(DISTINCT sr.type)[..5] AS sample_types
    ORDER BY requests DESC
""")
display(spark.createDataFrame(district_requests))

In [ ]:
# Assets in poor/critical condition with open service requests
at_risk_assets = run_cypher("""
    MATCH (sr:ServiceRequest)-[:AFFECTS]->(a:Asset)-[:IN_DISTRICT]->(d:District)
    WHERE a.condition IN ['poor', 'critical']
    RETURN a.asset_id AS asset, a.type AS asset_type, a.condition AS condition,
           d.name AS district, count(sr) AS open_requests
    ORDER BY open_requests DESC
    LIMIT 20
""")
display(spark.createDataFrame(at_risk_assets))

### K-12 Early Warning — At-risk student identification

In [ ]:
# High-risk students with interventions
at_risk = run_cypher("""
    MATCH (s:K12Student)-[:ATTENDS]->(sc:School)
    WHERE s.risk_score > 70
    OPTIONAL MATCH (s)-[:RECEIVED]->(i:Intervention)
    RETURN s.student_id AS student, s.grade_level AS grade,
           s.risk_score AS risk_score, s.attendance_rate AS attendance,
           s.gpa AS gpa, sc.name AS school,
           collect(i.type) AS interventions
    ORDER BY s.risk_score DESC
    LIMIT 25
""")
display(spark.createDataFrame(at_risk))

In [ ]:
# Schools with highest average risk scores
school_risk = run_cypher("""
    MATCH (s:K12Student)-[:ATTENDS]->(sc:School)
    RETURN sc.name AS school, sc.type AS school_type,
           round(avg(s.risk_score), 1) AS avg_risk,
           round(avg(s.attendance_rate), 1) AS avg_attendance,
           count(s) AS student_count,
           sc.title_i AS title_i
    ORDER BY avg_risk DESC
    LIMIT 15
""")
display(spark.createDataFrame(school_risk))

### Procurement — Vendor network analysis

In [ ]:
# Top vendors by contract value
top_vendors = run_cypher("""
    MATCH (ag:ProcAgency)-[:ISSUED]->(ct:Contract)-[:AWARDED_TO]->(v:ProcVendor)
    RETURN v.name AS vendor, v.minority_owned AS mbe, v.small_business AS sbe,
           count(ct) AS contracts, round(sum(ct.amount)) AS total_value
    ORDER BY total_value DESC
    LIMIT 20
""")
display(spark.createDataFrame(top_vendors))

In [ ]:
# Lobbyist → Vendor → Contract connections
lobby_contracts = run_cypher("""
    MATCH (lb:Lobbyist)-[:REPRESENTS]->(v:ProcVendor)<-[:AWARDED_TO]-(ct:Contract)<-[:ISSUED]-(ag:ProcAgency)
    RETURN lb.name AS lobbyist, lb.firm AS firm,
           v.name AS vendor, ag.name AS agency,
           ct.amount AS contract_value, ct.method AS procurement_method
    ORDER BY ct.amount DESC
    LIMIT 20
""")
display(spark.createDataFrame(lobby_contracts))

### Case Management (HHS) — Client service analysis

In [ ]:
# Clients enrolled in multiple programs
multi_program = run_cypher("""
    MATCH (cl:Client)-[:HAS_CASE]->(cs:Case)-[:ENROLLED_IN]->(pg:HHSProgram)
    WITH cl, collect(DISTINCT pg.name) AS programs, count(DISTINCT cs) AS cases
    WHERE size(programs) > 1
    RETURN cl.client_id AS client, cl.income_bracket AS income,
           cl.household_size AS household_size,
           cases, programs
    ORDER BY size(programs) DESC
    LIMIT 20
""")
display(spark.createDataFrame(multi_program))

In [ ]:
# Caseworker caseload analysis
caseload = run_cypher("""
    MATCH (cs:Case)-[:MANAGED_BY]->(cw:Caseworker)-[:WORKS_AT]->(ag:HHSAgency)
    RETURN cw.name AS caseworker, cw.specialization AS specialization,
           ag.name AS agency, count(cs) AS active_cases,
           round(avg(cs.benefit_amount), 2) AS avg_benefit
    ORDER BY active_cases DESC
    LIMIT 20
""")
display(spark.createDataFrame(caseload))

---
## 13. SLED PostgreSQL — JDBC Reads

Read SLED relational tables from PostgreSQL via JDBC.

In [ ]:
def read_sled_table(table_name):
    """Read a SLED table from PostgreSQL via JDBC."""
    return (
        spark.read
        .format("jdbc")
        .option("url", POSTGRES_JDBC)
        .option("dbtable", table_name)
        .option("user", POSTGRES_USER)
        .option("password", POSTGRES_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .load()
    )

In [ ]:
# Student enrollment — enrollment events with joins
df_enrollment = read_sled_table("sled_enrollment_events")
df_students = read_sled_table("sled_students")

print(f"Enrollment events: {df_enrollment.count():,}")
print(f"Students: {df_students.count():,}")

# Top departments by enrollment volume
display(
    df_enrollment
    .filter(col("action") == "enroll")
    .groupBy("semester")
    .agg(
        count("*").alias("enrollments"),
        countDistinct("student_id").alias("unique_students"),
    )
    .orderBy("semester")
)

In [ ]:
# K-12 — at-risk students by school
df_k12 = read_sled_table("sled_k12_students")
df_schools = read_sled_table("sled_schools")

display(
    df_k12
    .filter(col("risk_score") > 70)
    .join(df_schools, "school_id")
    .groupBy("name", "type")
    .agg(
        count("*").alias("high_risk_students"),
        avg("risk_score").alias("avg_risk_score"),
        avg("attendance_rate").alias("avg_attendance"),
    )
    .orderBy(col("high_risk_students").desc())
)

In [ ]:
# Procurement — contract analysis
df_contracts = read_sled_table("sled_contracts")

display(
    df_contracts
    .groupBy("category", "method")
    .agg(
        count("*").alias("contracts"),
        sum("amount").alias("total_value"),
        avg("amount").alias("avg_value"),
    )
    .orderBy(col("total_value").desc())
)

In [ ]:
# Case management — benefit analysis by program
df_cases = read_sled_table("sled_cases")

display(
    df_cases
    .groupBy("program", "determination")
    .agg(
        count("*").alias("cases"),
        avg("benefit_amount").alias("avg_benefit"),
        sum("benefit_amount").alias("total_benefits"),
    )
    .orderBy(col("total_benefits").desc())
)

In [ ]:
# Budget — expenditure analysis
df_budget = read_sled_table("sled_budget_transactions")

display(
    df_budget
    .filter(col("transaction_type") == "expenditure")
    .groupBy("fund_category", "fiscal_year")
    .agg(
        count("*").alias("transactions"),
        sum("amount").alias("total_spent"),
    )
    .orderBy("fiscal_year", col("total_spent").desc())
)

In [ ]:
# Citizen services — request resolution analysis
df_requests = read_sled_table("sled_service_requests")

display(
    df_requests
    .groupBy("request_type", "priority")
    .agg(
        count("*").alias("requests"),
        avg("response_time_hours").alias("avg_response_hours"),
        avg("satisfaction_rating").alias("avg_satisfaction"),
    )
    .orderBy(col("requests").desc())
)

---
## 14. Neo4j — General Graph Queries

Query the Neo4j graph database over Bolt (TLS via nginx on port 7688).

In [ ]:
# Test connectivity
print(run_cypher("RETURN 1 AS test"))

# Get database stats
stats = run_cypher("""
    CALL db.labels() YIELD label
    RETURN label, count(*) AS count
""")
print(f"\nLabels: {stats}")

In [ ]:
# Relationship overview
rel_overview = run_cypher("""
    MATCH ()-[r]->()
    RETURN type(r) AS relationship_type, count(r) AS count
    ORDER BY count DESC
""")
display(spark.createDataFrame(rel_overview))

---
## 15. PostgreSQL — JDBC Read

Read tables from the PostgreSQL database. Requires the PostgreSQL JDBC driver on your cluster (typically pre-installed on Databricks).

In [ ]:
# PostgreSQL with native SSL on port 15433
df_events = (
    spark.read
    .format("jdbc")
    .option("url", POSTGRES_JDBC)
    .option("dbtable", "kafka_event_logs")
    .option("user", POSTGRES_USER)
    .option("password", POSTGRES_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(df_events)

---
## 16. REST API — Service Health Checks

Test connectivity to all services through the API.

In [ ]:
checks = {
    "PostgreSQL ORM":  "/data-sources/orm",
    "PostgreSQL SQL":  "/data-sources/raw/sql",
    "Neo4j Health":    "/data-sources/neo4j/health",
    "Neo4j Version":   "/data-sources/neo4j/version",
    "Redis":           "/redis/test",
    "Kafka Events":    "/kafka/events?limit=1",
}

for name, path in checks.items():
    try:
        r = requests.get(f"{API_URL}{path}", headers=HEADERS, timeout=10)
        status = r.json().get("status", r.status_code)
        print(f"  {name}: {status}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

---
## 17. Cleanup

Stop all running generators, clear SLED data, and close connections.

In [ ]:
# Stop all running generators (core + SLED)
generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
for g in generators:
    if g["status"] == "running":
        requests.post(f"{API_URL}/kafka/generators/{g['generator_id']}/stop", headers=HEADERS)
        print(f"Stopped {g['use_case']} generator {g['generator_id']}")

# Clean up finished generators
cleanup = requests.delete(f"{API_URL}/kafka/generators/cleanup", headers=HEADERS).json()
print(f"Cleaned up {cleanup.get('removed', 0)} generator(s)")

# Close Neo4j driver
driver.close()
print("Neo4j driver closed")

In [ ]:
# Optional: Clear all SLED data from Neo4j and Postgres
# Uncomment to wipe SLED data after your session

# print("Clearing SLED Neo4j data...")
# for use_case in SLED_USE_CASES:
#     resp = requests.delete(f"{API_URL}/data-sources/neo4j/sled/{use_case}/clear", headers=HEADERS)
#     print(f"  {use_case}: {resp.json()}")

# print("\nClearing SLED Postgres data...")
# for use_case in SLED_USE_CASES:
#     resp = requests.delete(f"{API_URL}/data-sources/sled/{use_case}/clear", headers=HEADERS)
#     print(f"  {use_case}: {resp.json()}")

# Clear all checkpoints
# sled_checkpoints = [
#     "fraud_preview", "telemetry_preview", "web_preview",
#     "enrollment_preview", "budget_preview", "citizen_services_preview",
#     "k12_preview", "procurement_preview", "case_management_preview",
# ]
# for cp in sled_checkpoints:
#     dbutils.fs.rm(f"{CHECKPOINT_BASE}/{cp}", recurse=True)
# print("Checkpoints cleared")